# OpenEconIndex Analysis

This notebook analyzes the OpenEconIndex dataset, which maps WildChat conversations
to O*NET occupational tasks. It reproduces the figures from the paper, including:

- Figure 3: Occupational representation (Open Econ Index vs US Economy)
- Figure 4: Depth of AI usage by occupation
- Figure 5: Top skills represented in conversations
- Figure 15: Work activity coverage

## Setup

The dataset is loaded from HuggingFace (`umich-fatml/OpenEconIndex`), which contains
WildChat conversations with attached O*NET task mappings (both task text and task IDs).

Set your HuggingFace token via the `HF_TOKEN` environment variable before running.


In [ ]:
import json
import math
import os
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from datasets import load_dataset

ONET_DATA_DIR = '../onet_db'
DATASET_NAME = 'umich-fatml/OpenEconIndex'

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    print('Warning: HF_TOKEN env var not set. Loading may fail for private datasets.')


## Load OpenEconIndex Dataset

The dataset has the following relevant columns per row:
- `conversation_hash` — unique conversation ID
- `conversation` — list of message turns
- `request` — first user message summary
- `relevance` — occupational relevance flag (`y`/`n`)
- `matched_tasks` — list of `{task_id, task_text}` from unfiltered task matching
- `matched_tasks_filtered` — list of `{task_id, task_text}` from filtered task matching
- `matched_summary` — summary used for embedding-based task matching


In [ ]:
dataset = load_dataset(DATASET_NAME, split='train', token=HF_TOKEN)
print(f'Total rows: {len(dataset):,}')
print(f'Columns: {dataset.column_names}')


In [ ]:
# US-only conversation hashes (optional filter for figure 3)
# This file ships with the repo if you need to reproduce the US-only analysis.
US_HASH_FILE = 'us-hash.json'
us_hashes = set()
if os.path.exists(US_HASH_FILE):
    with open(US_HASH_FILE, 'r') as f:
        us_hashes = set(json.load(f))
    print(f'US conversations: {len(us_hashes):,}')
else:
    print(f'No us-hash.json found; skipping US-only analysis.')


## Build Chat -> Task Maps

Build the per-chat task lookup from the dataset's `matched_tasks_filtered` column
(use `matched_tasks` for unfiltered analysis). Each entry stores both task text
and task ID, so downstream joins to O*NET are stable across releases.


In [ ]:
chat_task_map_us = {}
chat_task_map_all = {}
task_count_dist_us = Counter()
task_count_dist_all = Counter()

for row in tqdm(dataset, desc='Building chat->task maps'):
    cid = row['conversation_hash']
    matched = row.get('matched_tasks_filtered') or []
    n_tasks = len(matched)
    task_count_dist_all[n_tasks] += 1

    summary = row.get('matched_summary', '') or row.get('request', '')

    if n_tasks > 0:
        chat_task_map_all[cid] = {
            'Chat': summary,
            'Tasks': [t['task_text'] for t in matched],
            'TaskIDs': [t['task_id'] for t in matched],
        }
    if cid in us_hashes:
        task_count_dist_us[n_tasks] += 1
        if n_tasks > 0:
            chat_task_map_us[cid] = {
                'Chat': summary,
                'Tasks': [t['task_text'] for t in matched],
                'TaskIDs': [t['task_id'] for t in matched],
            }

print(f'\nTask distribution (US chats):')
for n in sorted(task_count_dist_us.keys()):
    pct = task_count_dist_us[n] / max(sum(task_count_dist_us.values()), 1) * 100
    print(f'  {n} tasks: {task_count_dist_us[n]:,} ({pct:.1f}%)')
print(f'US chats with >=1 task: {len(chat_task_map_us):,}')

print(f'\nTask distribution (all chats):')
for n in sorted(task_count_dist_all.keys()):
    pct = task_count_dist_all[n] / max(sum(task_count_dist_all.values()), 1) * 100
    print(f'  {n} tasks: {task_count_dist_all[n]:,} ({pct:.1f}%)')
print(f'All chats with >=1 task: {len(chat_task_map_all):,}')


## Load O*NET Data and Define Mapping Functions

We use task IDs (when available) to look up the O*NET-SOC code for each task,
falling back to text matching for tasks not present in the lookup.


In [ ]:
def load_onet_file(filename):
    path = os.path.join(ONET_DATA_DIR, filename)
    if not os.path.exists(path):
        print(f'Warning: {filename} not found in {ONET_DATA_DIR}.')
        return pd.DataFrame()
    return pd.read_csv(path, sep='\t')

def prepare_onet_data():
    print('Loading O*NET data...')
    tasks_df = load_onet_file('Task Statements.txt')
    occ_df = load_onet_file('Occupation Data.txt')
    skills_df = load_onet_file('Skills.txt')
    return tasks_df, occ_df, skills_df


def map_chats_to_soc(user_data, tasks_df, mode='majority_vote'):
    """
    Match user task strings to O*NET SOC codes.
    Modes:
      - 'majority_vote': Assign chat to SOC with most matching tasks. Weight=1.0
      - 'first_task':    Assign chat to SOC of the first matched task. Weight=1.0
      - 'fractional':    Split chat into multiple rows by task distribution. Weight<1.0
    """
    print(f'Mapping Chats to SOC Codes (Mode: {mode})...')

    # Prefer task ID lookup (stable across O*NET releases)
    id_lookup = dict(zip(tasks_df['Task ID'].astype(str), tasks_df['O*NET-SOC Code']))
    text_lookup = dict(zip(tasks_df['Task'], tasks_df['O*NET-SOC Code']))

    mapped_rows = []

    for cid, entry in user_data.items():
        found = []  # (task_text, soc)
        task_ids = entry.get('TaskIDs') or [None] * len(entry['Tasks'])
        for task_str, tid in zip(entry['Tasks'], task_ids):
            soc = None
            if tid is not None:
                soc = id_lookup.get(str(tid))
            if soc is None:
                soc = text_lookup.get(task_str)
            if soc:
                found.append((task_str, soc))

        if not found:
            continue

        if mode == 'majority_vote':
            soc_counts = {}
            for _, soc in found:
                soc_counts[soc] = soc_counts.get(soc, 0) + 1
            best_soc = max(soc_counts, key=soc_counts.get)
            mapped_rows.append({
                'Chat_ID': cid,
                'O*NET-SOC Code': best_soc,
                'Matched_Tasks': [t[0] for t in found],
                'Task_Count': len(found),
                'Weight': 1.0,
            })
        elif mode == 'first_task':
            first_task, first_soc = found[0]
            mapped_rows.append({
                'Chat_ID': cid,
                'O*NET-SOC Code': first_soc,
                'Matched_Tasks': [t[0] for t in found],
                'Task_Count': len(found),
                'Weight': 1.0,
            })
        elif mode == 'fractional':
            total = len(found)
            soc_groups = {}
            for task_str, soc in found:
                soc_groups.setdefault(soc, []).append(task_str)
            for soc, tasks in soc_groups.items():
                mapped_rows.append({
                    'Chat_ID': cid,
                    'O*NET-SOC Code': soc,
                    'Matched_Tasks': tasks,
                    'Task_Count': len(tasks),
                    'Weight': len(tasks) / total,
                })

    return pd.DataFrame(mapped_rows)


def get_top_occupations(mapped_df, tasks_df, occ_df, top_n=10, min_unique_tasks=5):
    print(f'\nCalculating Top {top_n} Occupations (Min Unique Tasks: {min_unique_tasks})...')
    task_lookup = dict(zip(tasks_df['Task'], tasks_df['O*NET-SOC Code']))

    soc_observed_tasks = {}
    for _, row in mapped_df.iterrows():
        soc = row['O*NET-SOC Code']
        valid = {t for t in row['Matched_Tasks'] if task_lookup.get(t) == soc}
        soc_observed_tasks.setdefault(soc, set()).update(valid)

    valid_socs = [s for s, tasks in soc_observed_tasks.items() if len(tasks) >= min_unique_tasks]
    print(f' -> {len(valid_socs)} occupations passed the {min_unique_tasks}-task filter.')
    if not valid_socs:
        return pd.DataFrame()

    filtered = mapped_df[mapped_df['O*NET-SOC Code'].isin(valid_socs)]
    usage = filtered.groupby('O*NET-SOC Code')['Weight'].sum()
    total_per_soc = tasks_df.groupby('O*NET-SOC Code')['Task'].nunique()

    analysis = pd.DataFrame({
        'Raw_Chat_Weight': usage,
        'Total_ONET_Tasks': total_per_soc,
    }).dropna()
    analysis['Normalized_Score'] = analysis['Raw_Chat_Weight'] / analysis['Total_ONET_Tasks']
    analysis = analysis.merge(occ_df[['O*NET-SOC Code', 'Title']], on='O*NET-SOC Code', how='left')
    top = analysis.sort_values('Normalized_Score', ascending=False).head(top_n).reset_index(drop=True)
    top.index += 1
    return top[['Title', 'O*NET-SOC Code', 'Normalized_Score', 'Raw_Chat_Weight', 'Total_ONET_Tasks']]


def save_fig(name):
    plt.savefig(f'{name}.pdf')
    plt.savefig(f'{name}.png', dpi=200)
    plt.close()


## Figures 3, 4, 5

In [ ]:
def plot_figure_3(mapped_df, occ_df):
    print('Generating Figure 3...')
    mapped_df['Major_Group'] = mapped_df['O*NET-SOC Code'].str[:2]
    soc_names = {
        '11': 'Management', '13': 'Business & Financial', '15': 'Computer & Mathematical',
        '17': 'Architecture & Engineering', '19': 'Life, Physical, Social Science',
        '21': 'Community & Social Service', '23': 'Legal', '25': 'Education & Library',
        '27': 'Arts, Design, Entertainment', '29': 'Healthcare Practitioners',
        '31': 'Healthcare Support', '33': 'Protective Service', '35': 'Food Prep & Serving',
        '37': 'Building Cleaning', '39': 'Personal Care', '41': 'Sales',
        '43': 'Office & Admin Support', '45': 'Farming, Fishing', '47': 'Construction',
        '49': 'Installation & Repair', '51': 'Production', '53': 'Transportation'
    }
    group_weights = mapped_df.groupby('Major_Group')['Weight'].sum()
    chat_dist = (group_weights / mapped_df['Weight'].sum()).rename('Chat_Pct')
    us_dist = pd.Series({
        '11': 0.069, '13': 0.066, '15': 0.034, '17': 0.017, '19': 0.009,
        '21': 0.02, '23': 0.016, '25': 0.058, '27': 0.014, '29': 0.061,
        '31': 0.047, '33': 0.023, '35': 0.09, '37': 0.03, '39': 0.02,
        '41': 0.088, '43': 0.122, '45': 0.03, '47': 0.041, '49': 0.039,
        '51': 0.058, '53': 0.091
    }, name='US_Pct')
    df = pd.concat([chat_dist, us_dist], axis=1).fillna(0)
    df['Group_Name'] = df.index.map(soc_names)
    df = df.dropna(subset=['Group_Name']).sort_values('Chat_Pct', ascending=False)
    plt.figure(figsize=(10, 8))
    y_range = range(len(df))
    plt.hlines(y=y_range, xmin=df['US_Pct'], xmax=df['Chat_Pct'], color='grey', alpha=0.4)
    plt.scatter(df['Chat_Pct'], y_range, color='tomato', label='Open Econ Index', s=100)
    plt.scatter(df['US_Pct'], y_range, color='navy', label='US Economy', s=100)
    plt.yticks(y_range, df['Group_Name'])
    plt.xlabel('Representation')
    plt.legend()
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    save_fig('figure_3_representation')


def plot_figure_4(mapped_df, tasks_df, min_chats=math.floor(15*750000/1e6)):
    print(f'Generating Figure 4 (Threshold: {min_chats} chats)...')
    total_per_soc = tasks_df.groupby('O*NET-SOC Code')['Task'].nunique()
    all_found = mapped_df.explode('Matched_Tasks')['Matched_Tasks']
    counts = all_found.value_counts()
    used = counts[counts >= min_chats].index.tolist()
    used_df = tasks_df[tasks_df['Task'].isin(used)]
    used_per_soc = used_df.groupby('O*NET-SOC Code')['Task'].nunique()
    coverage = pd.concat([total_per_soc, used_per_soc], axis=1)
    coverage.columns = ['Total_Tasks', 'Used_Tasks']
    coverage = coverage.fillna(0)
    coverage['Fraction'] = (coverage['Used_Tasks'] / coverage['Total_Tasks']).clip(upper=1.0)
    x = np.sort(coverage['Fraction'])
    y = 1.0 - np.arange(len(x)) / len(x)
    plt.figure(figsize=(8, 6))
    plt.plot(x, y, marker='.', linestyle='-')
    plt.xlabel('Minimum Fraction of Tasks in Use')
    plt.ylabel('Fraction of Occupations')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xlim(0, 1.0)
    plt.ylim(0, 1.05)
    for thresh in [0.25, 0.50, 0.75]:
        if len(x) > 0:
            idx = (np.abs(x - thresh)).argmin()
            if abs(x[idx] - thresh) < 0.1:
                val = y[idx]
                plt.plot(x[idx], val, 'ro')
                plt.annotate(f'{val:.1%} of occs > {thresh*100}% tasks',
                             (x[idx], val), xytext=(10, 10), textcoords='offset points')
    plt.tight_layout()
    save_fig('figure_4_depth')


def plot_figure_5(mapped_df, skills_df):
    print('Generating Figure 5...')
    important = skills_df[(skills_df['Scale ID'] == 'IM') & (skills_df['Data Value'] > 4.0)]
    soc_weights = mapped_df.groupby('O*NET-SOC Code')['Weight'].sum().reset_index()
    soc_weights.columns = ['O*NET-SOC Code', 'Total_Weight']
    merged = pd.merge(important, soc_weights, on='O*NET-SOC Code', how='inner')
    total_weight = mapped_df['Weight'].sum()
    prevalence = merged.groupby('Element Name')['Total_Weight'].sum() / total_weight
    top = prevalence.sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 8))
    sns.barplot(x=top.values * 100, y=top.index, palette='viridis')
    plt.xlabel('Percentage of Total Conversations')
    plt.ylabel('Skill')
    plt.tight_layout()
    save_fig('figure_5_skills')


In [ ]:
tasks_df, occ_df, skills_df = prepare_onet_data()
if not tasks_df.empty:
    mapped_df_us = map_chats_to_soc(chat_task_map_us, tasks_df, mode='fractional') if chat_task_map_us else pd.DataFrame()
    mapped_df_all = map_chats_to_soc(chat_task_map_all, tasks_df, mode='fractional')

    print(f'US mapped: {len(mapped_df_us):,} rows from {mapped_df_us["Chat_ID"].nunique() if len(mapped_df_us) else 0:,} chats')
    print(f'All mapped: {len(mapped_df_all):,} rows from {mapped_df_all["Chat_ID"].nunique():,} chats')

    if len(mapped_df_us) > 0:
        plot_figure_3(mapped_df_us, occ_df)
    plot_figure_4(mapped_df_all, tasks_df)
    plot_figure_5(mapped_df_all, skills_df)
    print('All figures generated.')


In [ ]:
top_occs = get_top_occupations(mapped_df_all, tasks_df, occ_df, top_n=20, min_unique_tasks=5)
print('\nTop 20 Occupations by Normalized Usage Intensity:')
print(top_occs.to_string())


## Wage vs AI Usage

In [ ]:
# --- Wage vs AI Usage scatter plot ---
WAGE_FILE = os.path.join(ONET_DATA_DIR, 'wage_data.csv')
if os.path.exists(WAGE_FILE):
    wage_df = pd.read_csv(WAGE_FILE)
    print(f'Wage data: {len(wage_df)} occupations')

    soc_usage = mapped_df_all.groupby('O*NET-SOC Code')['Weight'].sum().reset_index()
    soc_usage.columns = ['O*NET-SOC Code', 'AI_Conversations']
    total = soc_usage['AI_Conversations'].sum()
    soc_usage['AI_Pct'] = soc_usage['AI_Conversations'] / total * 100

    merged = pd.merge(wage_df, soc_usage, on='O*NET-SOC Code', how='inner')
    plt.figure(figsize=(10, 6))
    plt.scatter(merged['Annual_Wage'], merged['AI_Pct'], alpha=0.5)
    plt.xlabel('Annual Wage (USD)')
    plt.ylabel('AI Conversation Share (%)')
    plt.tight_layout()
    save_fig('figure_wage_vs_ai_usage')
else:
    print(f'No wage_data.csv found at {WAGE_FILE}; skipping wage analysis.')


## Figure 15: Work Activity Coverage

In [ ]:
tasks_to_dwas_df = load_onet_file('Tasks to DWAs.txt')
dwa_ref_df = load_onet_file('DWA Reference.txt')
wa_df = load_onet_file('Work Activities.txt')

def plot_figure_15_wa(mapped_df, tasks_df, tasks_to_dwas_df, dwa_ref_df, wa_df, top_n=None):
    """
    For each O*NET Work Activity (GWA), compute:
      - n unique chats covering at least one task in that WA
      - chat coverage = covered_chats / total_chats
    Returns a series sorted descending.
    """
    # task -> list of GWA element ids
    task_to_dwa = tasks_to_dwas_df[['Task ID', 'DWA ID']].drop_duplicates()
    dwa_to_iwa = dwa_ref_df[['DWA ID', 'IWA ID']].drop_duplicates()
    wa_lookup = wa_df[['Element ID', 'Element Name']].drop_duplicates()
    iwa_to_gwa = dwa_ref_df[['IWA ID', 'Element ID']].drop_duplicates()

    # task_id -> WA name
    merged = task_to_dwa.merge(dwa_to_iwa, on='DWA ID').merge(iwa_to_gwa, on='IWA ID').merge(
        wa_lookup, on='Element ID')
    task_to_wa = merged.groupby('Task ID')['Element Name'].apply(set).to_dict()
    text_to_id = dict(zip(tasks_df['Task'], tasks_df['Task ID'].astype(str)))

    chat_to_wa = {}
    for _, row in mapped_df.iterrows():
        cid = row['Chat_ID']
        wa_set = chat_to_wa.setdefault(cid, set())
        for t in row['Matched_Tasks']:
            tid = text_to_id.get(t)
            if tid and str(tid) in {str(k) for k in task_to_wa.keys()}:
                wa_set.update(task_to_wa[int(tid) if tid.isdigit() else tid])

    total_chats = len(chat_to_wa)
    wa_counts = Counter()
    for wa_set in chat_to_wa.values():
        for wa in wa_set:
            wa_counts[wa] += 1

    wa_coverage = pd.Series({wa: count / total_chats for wa, count in wa_counts.items()})
    wa_coverage = wa_coverage.sort_values(ascending=False)
    if top_n:
        wa_coverage = wa_coverage.head(top_n)

    plt.figure(figsize=(10, max(6, len(wa_coverage) * 0.3)))
    sns.barplot(x=wa_coverage.values, y=wa_coverage.index, palette='viridis')
    plt.xlabel('Fraction of Chats Covering Activity')
    plt.tight_layout()
    save_fig('figure_15_work_activities')
    return wa_coverage

wa_coverage = plot_figure_15_wa(mapped_df_all, tasks_df, tasks_to_dwas_df, dwa_ref_df, wa_df, top_n=24)
print('\nTop 10 Work Activities by Chat Coverage:')
print(wa_coverage.sort_values(ascending=False).head(10))


## Pipeline Examples

Sample conversations with their matched tasks and SOC codes (for figures
showing concrete pipeline outputs).


In [ ]:
task_lookup = dict(zip(tasks_df['Task'], tasks_df['O*NET-SOC Code']))
title_lookup = dict(zip(occ_df['O*NET-SOC Code'], occ_df['Title']))

FORCE_INCLUDE = {'cf1267ca6b2f6fccc9c36652a00059a1'}

# Pre-select candidate hashes from the dataset
example_candidates = set()
example_info = {}
for cid, entry in chat_task_map_all.items():
    tasks = entry['Tasks']
    summary = entry['Chat']
    forced = cid in FORCE_INCLUDE
    if not forced:
        if len(tasks) < 2 or len(tasks) > 3:
            continue
        if len(summary) > 100 or len(summary) < 20:
            continue
    socs = [task_lookup.get(t) for t in tasks if task_lookup.get(t)]
    if not socs:
        continue
    if not forced and len(set(socs)) != 1:
        continue
    primary_soc = max(set(socs), key=socs.count)
    example_candidates.add(cid)
    example_info[cid] = {
        'summary': summary, 'tasks': tasks,
        'soc': primary_soc, 'title': title_lookup.get(primary_soc, '?'),
    }

print(f'Found {len(example_candidates):,} candidate hashes')

# Walk the dataset to grab example conversations (diverse occupations)
MAX_EXAMPLES = 20
examples = []
seen_socs = set()
scanned = 0
for row in dataset:
    scanned += 1
    h = row['conversation_hash']
    if h not in example_candidates:
        continue
    cand = example_info[h]
    if cand['soc'] in seen_socs and h not in FORCE_INCLUDE:
        continue
    turns = row['conversation']
    total_chars = sum(len(t['content']) for t in turns)
    if total_chars > 2000 or len(turns) > 4:
        if h not in FORCE_INCLUDE:
            continue
    user_msg = next((t['content'] for t in turns if t['role'] == 'user'), '')
    asst_msg = next((t['content'] for t in turns if t['role'] == 'assistant'), '')
    examples.append({
        'hash': h, 'user_msg': user_msg[:300], 'asst_msg': asst_msg[:300],
        'summary': cand['summary'], 'tasks': cand['tasks'],
        'soc': cand['soc'], 'title': cand['title'],
    })
    seen_socs.add(cand['soc'])
    if len(examples) >= MAX_EXAMPLES:
        break

print(f'Found {len(examples)} examples after scanning {scanned:,} rows\n')

for i, ex in enumerate(examples, 1):
    print('=' * 80)
    print(f"Example {i}: {ex['title']} ({ex['soc']})")
    print(f"  User: {ex['user_msg']}")
    print(f"  Assistant: {ex['asst_msg'][:200]}...")
    print(f"  Summary: {ex['summary']}")
    for j, t in enumerate(ex['tasks']):
        print(f'  Task {j+1}: {t}')
    print()
